In [10]:
import argparse
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
from tqdm import tqdm

import pybacktest
from pybacktest.bookcore import BookCore



In [11]:
data_dir = Path(r'E:\tmp\OKX-Books-BTC-USDT-400')
pfs = list(data_dir.glob("*.parquet"))

In [12]:
df_back = pq.ParquetFile(pfs[0]).read().to_pandas()

In [16]:
df = df_back.copy()
tmp = df[df['action'] == 'snapshot'][['ts', 'action']]

In [37]:
tss = np.append(tmp.index.values, df.index.values[-1])
all(tss[i] + 100_000 >=tss[i + 1] for i in range(len(tss) - 1))

True

In [41]:
tmp = df[df['action'] == 'asdf'].index
tmp.empty

True

In [9]:
interval = 100000
bc = None
iter_rows = 0
df = df_back.copy()
# dps = list(map(convert, df.to_dict(orient='records')))
dps = df.to_dict(orient='records')
for dp in dps:
    if bc == None:
        if dp['action'] != 'snapshot':
            continue
        instId = dp['arg']['instId']
        bc = BookCore(instId)
    
    try: 
        bc.set_datapoint(dp)
    except Exception as e:
        print(f"Error processing datapoint: {e}")
        print(f"Datapoint: {dp}")
        raise e
    iter_rows += 1

    if iter_rows >= interval: # should insert a snapshot
        asks_bl = [[str(bl.price),str(bl.amount),'0',str(bl.count)] for bl in bc.asks]
        bids_bl = [[str(bl.price),str(bl.amount),'0',str(bl.count)] for bl in bc.bids]
        # update the row with the new snapshot data
        dp['data']['asks'] = asks_bl
        dp['data']['bids'] = bids_bl
        dp['action'] = 'snapshot'
        iter_rows = 0
        break
table = pa.Table.from_pylist(dps)

In [7]:
def convert(datapoint):
    new_data = {
        'asks': [bl.tolist() for bl in datapoint['data']['asks']],
        'bids': [bl.tolist() for bl in datapoint['data']['bids']],
        'checksum': datapoint['data']['checksum'],
        'prevSeqId': datapoint['data']['prevSeqId'],
        'seqId': datapoint['data']['seqId'],
        'ts': datapoint['data']['ts']
    }
    return {
        'arg': datapoint['arg'],
        'data': new_data,
        'action': datapoint['action'],
        'ts': datapoint['ts']
    }
list(map(convert, df.to_dict('records')))[:10]

KeyboardInterrupt: 

In [ ]:
df_back[df_back['action'] == 'snapshot'].index

Index([233285, 391568, 657489, 821565, 843269, 848913, 852340, 951367], dtype='int64')